# Przewodnik po konkursowym pakiecie powitalnym

Aby łatwiej zacząć, w *pakiecie powitalnym* dostajesz gotowe narzędzia, zawarte w dwóch plikach Pythona:

* `flu_get_data.py` - moduł zawierający funkcje do pobierania z internetu danych, na których ma się opierać model, 
* `flu_test.py` - moduł z funkcjami do obliczania błędu predykcji modelu.

Ponadto w *pakiecie powitalnym* są też pliki: 

* `zlotamaseczka.ipynb` - notatnik Jupytera zawierający przewodnik po w/w funkcjach (jest też dostępny na [Colabie](https://colab.research.google.com/drive/1aNas73Vjh4RmlfaF2iRnBNXMLbkZHD_z?usp=sharing)),
* `zlotamaseczka.pdf` - wersja PDF tegoż notatnika.

(Skoro właśnie czytasz ten tekst, to znaczy, że otworzyłaś jeden z powyższych dokumentów.)

Aby wziąć udział w konkursie, nie musisz korzystać z żadnego z plików zawartych w pakiecie. Ale jeśli chcesz - i wierzysz, że nie ma w nich błędów - możesz ich użyć dla własnej wygody.

Zaimportujmy w/w moduły Pythona, a także moduł Pandas.

In [1]:
import flu_get_data as fgd
import flu_test as ft
import pandas as pd

Pandas pozwala na wygodną pracę z danymi tabelarycznymi - arkuszami Excela i plikami CSV - i za chwilę to się nam przyda.

## Krok 1. Pobieranie danych
 
Zobaczmy, jak można użyć funkcji z modułu `flu_get_data` do wygenerowania danych wejściowych dla naszego modelu. Przyjmijmy, że interesuje nas województwo łódzkie. Ponieważ mamy prognozować grypę na początku 2027 roku, ustawiamy zmienną `year` na 2027: 

In [2]:
region = "ŁÓDZKIE" # w konkursie przyjmujemy, że nazwy województw są pisane WIELKIMI LITERAMI
year = 2027
weeks_forecast = [3, 4, 5, 6, 7] # w tych tygodniach mamy prognozować (numeracja ISO)


Zanim pobierzemy dane wejściowe dla modelu, musimy najpierw sprecyzować, z jakiego dokładnie okresu muszą pochodzić (w zadaniu mowa jest o pewnych tygodniach i miesiącach). Funkcja 

```python
weeks, months, days = fgd.get_data_params(year)
```

sama to ustali:

* wyznaczy numery (w standardzie ISO) czterech ostatnich tygodni roku poprzedniego (tzn. `year-1`, czyli u nas: 2026) i zapisze je do zmiennej `weeks`
* zwróci też listę `months`: numerów miesięcy roku `year-1`, z których należy zebrać dane o szczepieniach - w zadaniu zawsze wymaga się tych samych: od sierpnia do listopada
* określi, z jakich dni należy pobrać dane o pogodzie - pierwszy i ostatni znajdą się w `days`


In [3]:
weeks, months, days = fgd.get_data_params(year)

print(f'Prognoza: województwo {region}, rok {year}, tygodnie {weeks_forecast}')
print('Dane wejściowe:')
print(f'\tZachorowania: rok {year-1}, tygodnie {weeks}')
print(f'\tSzczepienia:  rok {year-1}, tygodnie {months}')
print(f'\tPogoda:       dni {days[0]} ... {days[1]}')


Prognoza: województwo ŁÓDZKIE, rok 2027, tygodnie [3, 4, 5, 6, 7]
Dane wejściowe:
	Zachorowania: rok 2026, tygodnie [50, 51, 52, 53]
	Szczepienia:  rok 2026, tygodnie [8, 9, 10, 11]
	Pogoda:       dni 2026-12-07 ... 2027-01-03


Teraz pobierzemy z serwisu [e-zdrowie](https://ezdrowie.gov.pl) pełny arkusz Excela zawierający dane zachorowaniach i szczepieniach. To może chwilę potrwać!

In [4]:
url = 'https://ezdrowie.gov.pl/pobierz/14553'
source_sheets  = pd.read_excel(url, sheet_name=["Tabela 3", "Tabela 5"], skiprows=5)


Używając funkcji `fetch_flu_data`, wyciągniemy stamtąd te dane o zachorowaniach i szczepieniach, które nas interesują - zapisując je w folderze `Data`:

In [5]:
df_infected, df_vaccinated = fgd.fetch_flu_data(source_sheets, year, region, 'data')
print('=== Zachorowania:\n', df_infected.head())
print('=== Szczepienia:\n', df_vaccinated.head())


Wybieram dane z arkusza Excela z portalu ezdrowie.gov.pl (mode = 'data')
	Zachorowania dla: rok 2026 tygodnie: [50, 51, 52, 53]
	Nie zapisuję: brak danych!
	Szczepienia dla: rok 2026 miesiące: [8, 9, 10, 11]
	Zapisuję do 2027-ŁÓDZKIE-vaccinated-data.csv
=== Zachorowania:
 Empty DataFrame
Columns: [Choroba, Rok, Numer tygodnia, Województwo, Liczba przypadków na 100k]
Index: []
=== Szczepienia:
       Choroba      Sezon    Miesiąc Kategoria wiekowa Województwo       Płeć  \
32385   Grypa  2026/2027 2026-08-01               0-4     ŁÓDZKIE    Kobieta   
32386   Grypa  2026/2027 2026-08-01               0-4     ŁÓDZKIE  Mężczyzna   
32419   Grypa  2026/2027 2026-08-01               5-9     ŁÓDZKIE    Kobieta   
32420   Grypa  2026/2027 2026-08-01               5-9     ŁÓDZKIE  Mężczyzna   
32453   Grypa  2026/2027 2026-08-01             10-14     ŁÓDZKIE    Kobieta   

       Liczba zaszczepionych  
32385                     23  
32386                     28  
32419                     42 

Hmmm.... Mamy problem! Nie wszystkie dane udało się pobrać... dlaczego?

No tak, przecież interesują nas dane o zachorowaniach z **grudnia 2026**, a tych danych jeszcze w listopadzie 2026 po prostu *nie ma*! 

Aby więc zapoznać się z funkcjonalnością modułu `flu_get_data`, spróbujmy powtórzyć to wszystko, co do tej pory - ale przyjmijmy tymczasowo, że interesuje nas prognoza na **2026** rok. Wtedy dane wejściowe będą pochodzić z 2025, więc uda się je wszystkie pobrać. 

Mało tego: jeśli *nasz model* działa, to będziemy również mogli - pobierając kolejne dane - sprawdzić, jak dobrze nasz model przewiduje zachorowania w 2026 (na podstawie danych z roku poprzedniego). To bardzo ważne, aby ocenić jakość modelu i zastanowić się, jakich wymaga poprawek.

In [6]:
region = "ŁÓDZKIE"
year = 2026 # UWAGA: dla testów tylko!
weeks_forecast = [3, 4, 5, 6, 7] # w tych tygodniach mamy prognozować (numeracja ISO)

weeks, months, days = fgd.get_data_params(year)

print('------ Parametry zadania ------')
print(f'Prognoza: województwo {region}, rok {year}, tygodnie {weeks_forecast}')
print('Dane wejściowe:')
print(f'\tZachorowania: rok {year-1}, tygodnie {weeks}')
print(f'\tSzczepienia:  rok {year-1}, tygodnie {months}')
print(f'\tPogoda:       dni {days[0]} ... {days[1]}')
print('--------------------------------')

df_infected, df_vaccinated = fgd.fetch_flu_data(source_sheets, year, region, 'data')
print('=== Zachorowania:\n', df_infected.head())
print('=== Szczepienia:\n', df_vaccinated.head())



------ Parametry zadania ------
Prognoza: województwo ŁÓDZKIE, rok 2026, tygodnie [3, 4, 5, 6, 7]
Dane wejściowe:
	Zachorowania: rok 2025, tygodnie [49, 50, 51, 52]
	Szczepienia:  rok 2025, tygodnie [8, 9, 10, 11]
	Pogoda:       dni 2025-12-01 ... 2025-12-28
--------------------------------
Wybieram dane z arkusza Excela z portalu ezdrowie.gov.pl (mode = 'data')
	Zachorowania dla: rok 2025 tygodnie: [49, 50, 51, 52]
	Zapisuję do 2026-ŁÓDZKIE-infected-data.csv
	Szczepienia dla: rok 2025 miesiące: [8, 9, 10, 11]
	Zapisuję do 2026-ŁÓDZKIE-vaccinated-data.csv
=== Zachorowania:
      Choroba   Rok  Numer tygodnia Województwo  Liczba przypadków na 100k
3908   Grypa  2025              49     ŁÓDZKIE                  20.832022
3924   Grypa  2025              50     ŁÓDZKIE                  46.785250
3940   Grypa  2025              51     ŁÓDZKIE                  75.733081
3956   Grypa  2025              52     ŁÓDZKIE                  79.682485
=== Szczepienia:
       Choroba      Sezon    Mie

Tak, teraz udało się pobrać wszystkie dane wejściowe z serwisu [e-zdrowie](https://ezdrowie.gov.pl). Zostały one zapisane do plików:

* `Data/2026-ŁÓDZKIE-infected-data.csv`
* `Data/2026-ŁÓDZKIE-vaccinated-data.csv`

Jak chcesz, możesz je teraz sobie otworzyć w dowolnym arkuszu kalkulacyjnym (np. w Excelu lub w LibreOffice Calc) i obejrzeć (ale najpierw przeczytaj wstawkę *CSV, a sprawa polska* poniżej). 

<div style="border: 2px solid #2b6cb0; 
    border-radius: 12px; 
    padding: 15px; 
    background-color: #faffeb; 
    color: #2d3748;
    margin: 10px 0;
    width: 80%;">

**CSV, a sprawa polska**

Korzystając z okazji, warto zdać sobie sprawę z ryzyk/potencjalnych kłopotów, z jakimi mogą spotkać się polscy użytkownicy plików CSV.

Powszechnym standardem zapisu danych tabelarycznych do plików CSV jest oddzielanie zapisywanych wartości przecinkiem (notabene, sam skrót CSV, ang. *comma-separated values*, oznacza właśnie *wartości rozdzielone przecinkiem*). Liczby rzeczywiste zapisuje się w nim z kropką jako separatorem dziesiętnym (np.`3.1415`).

Ponieważ jednak pechowo w Polsce norma wymaga zapisywania liczb z przecinkiem w roli separatora dziesiętnego (czyli `3,1415`) to w efekcie typowy *polski* CSV oddziela wartości... *średnikiem*. Łatwo domyślić się, że to może prowadzić do bałaganu. Dodatkowo jeszcze, *polskie litery* mogą też być zakodowane na różny sposób.

W naszym konkursie trzymamy się założenia, że pliki CSV z danymi będą zawsze miały

* wartości oddzielone *przecinkami*,
* liczby rzeczywiste zapisane z *kropką* jako separatorem dziesiętnym,
* kodowanie *UTF-8*.

Są to domyślne nastawy Pandas, zatem wczytanie takiego pliku, powiedzmy, `dane.csv`, do ramki Pandas będzie wyjątkowo proste, np.:

```python
df = pd.read_csv('dane.csv')
```

Z drugiej strony, *podgląd* zawartości takiego pliku CSV w polskich wersjach popularnych arkuszy kalkulacyjnych może wymagać nieco zachodu:

* LibreOffice Calc:
    * Otwórz plik `dane.csv`.
    * W oknie `Import tekstu`:
        * Jako `Zestaw znaków` wybierz UTF-8
        * Jako `Ustawienia regionalne` wybierz Angielski (UK)
        * Jako `Opcje separatora` zaznacz Przecinek (i odznacz średnik).
        * Kliknij OK.
* Microsoft Excel:
    * Zamiast otwierać plik dwukrotnym kliknięciem (co przy polskich ustawieniach zapewne scali kolumny):
        * Otwórz pusty arkusz w Excelu.
        * Przejdź do zakładki `Dane` > `Pobierz dane` > `Z pliku` > `Tekst/CSV`.
        * Wybierz plik `dane.csv`.
        * Upewnij się, że wybrano separator Przecinek oraz kodowanie UTF-8
        * Kliknij `Importuj`.

</div>

W zadaniu można jeszcze korzystać z danych o pogodzie. Aby je pobrać, wykorzystamy kolejną funkcję z modułu `flu_get_data`:  `fetch_weather_data`. Podobnie jak poprzednia funkcja, ona też zapisze pobrane dane do pliku CSV

* `Data/2026-ŁÓDZKIE-weather-data.csv`

i dodatkowo zwróci ramkę Pandas z tymi danymi.

In [7]:
df_weather = fgd.fetch_weather_data(year, region, days)
print('=== Pogoda:\n', df_weather.head())


Pobieram współrzędne dla miasta Łódź z portalu open-meteo.com
Pobieram dane pogodowe z portalu open-meteo.com
	Łódź (51.77058, 19.47395) | 2025-12-01 ... 2025-12-28
	Zapisuję do 2026-ŁÓDZKIE-weather-data.csv
=== Pogoda:
                time  temperature_2m (°C)  relative_humidity_2m (%)  \
0  2025-12-01T00:00                  0.7                        96   
1  2025-12-01T01:00                  0.7                        96   
2  2025-12-01T02:00                  0.5                        96   
3  2025-12-01T03:00                  0.5                        96   
4  2025-12-01T04:00                  0.3                        97   

   apparent_temperature (°C)  precipitation (mm)  weather_code (wmo code)  
0                       -2.2                 0.0                        3  
1                       -2.3                 0.0                        3  
2                       -2.6                 0.0                        3  
3                       -2.7                 0.0      

## Krok 2. Testowanie modelu

Przypuśćmy, że mamy już gotowy model - zadany przez funkcję `flu_predict` - na przykład taką, jak poniżej. Mając pobrane wszystkie dane wejściowe, możemy teraz wygenerować prognozę!

In [8]:
'''
To jest BARDZO SŁABY "model".
Na każdy z tygodni (3,4,5,6,7) roku year prognozuje to samo: maksymalną liczbę zachorowań (w województwie `region`, na 100 tys) odnotowaną w ostatnich czterech miesiącach roku `year-1`
'''

import pandas as pd

def flu_predict(year, region, infected, vaccinated, weather):
    # najpierw wczytujemy do Pandas dane z plików CSV
    df_infected_data = pd.read_csv(infected)
    df_vaccinated_data = pd.read_csv(vaccinated)
    df_weather_data = pd.read_csv(weather)
    # wyznaczamy prognozę
    infected_data_max = df_infected_data["Liczba przypadków na 100k"].max()
    forecast = [infected_data_max]*5
    return forecast

infected = 'Data/2026-ŁÓDZKIE-infected-data.csv'
vaccinated = 'Data/2026-ŁÓDZKIE-vaccinated-data.csv'
weather = 'Data/2026-ŁÓDZKIE-weather-data.csv'
forecast = flu_predict(2026, 'ŁÓDZKIE', infected, vaccinated, weather)

print('Prognoza zachorowań z modelu:\n', forecast)

Prognoza zachorowań z modelu:
 [79.68248526025923, 79.68248526025923, 79.68248526025923, 79.68248526025923, 79.68248526025923]


Aby sprawdzić, na ile było to skuteczne, musimy wiedzieć, ile *naprawdę* było zachorowań w tym czasie. Użyjemy znów funkcji `fetch_flu_data` - ale tym razem w trybie `forecast`, który spowoduje, że zostaną pobrane dane dla prognozowanego okresu.

In [9]:
df_infected_forecast, _ = fgd.fetch_flu_data(source_sheets, year, region, 'forecast')
ground_truth = df_infected_forecast['Liczba przypadków na 100k'].tolist()
print('Faktyczne zachorowania:\n', ground_truth)

Wybieram dane z arkusza Excela z portalu ezdrowie.gov.pl (mode = 'forecast')
	Zachorowania dla: rok 2026 tygodnie: [3, 4, 5, 6, 7]
	Zapisuję do 2026-ŁÓDZKIE-infected-forecast.csv
	Szczepienia dla: rok 2026 miesiące: [1, 2]
	Zapisuję do 2026-ŁÓDZKIE-vaccinated-forecast.csv
Faktyczne zachorowania:
 [172.6887847769997, 382.0506087941514, 617.1920603955047, 521.0175574887866, 290.30291062411436]


Korzystając z funkcji `flu_error` z modułu `flu_test` (który również jest w naszym pakiecie powitalnym), możemy policzyć błąd prognozy zgodnie z konkursowymi zasadami:

In [10]:
error = ft.flu_error(year, region, forecast, ground_truth, weeks_forecast)

ŁÓDZKIE 2026 tydz 03 | progn 79.68 | fakt 172.69 | różnica = -93.01
ŁÓDZKIE 2026 tydz 04 | progn 79.68 | fakt 382.05 | różnica = -302.37
ŁÓDZKIE 2026 tydz 05 | progn 79.68 | fakt 617.19 | różnica = -537.51
ŁÓDZKIE 2026 tydz 06 | progn 79.68 | fakt 521.02 | różnica = -441.34
ŁÓDZKIE 2026 tydz 07 | progn 79.68 | fakt 290.30 | różnica = -210.62

Błąd prognozy dla województwa ŁÓDZKIE w roku 2026 w tygodniach [3, 4, 5, 6, 7]: 1584.84


## Krok 3. Więcej danych (i więcej testów)

Aby ułatwić Ci dogłębne testowanie modelu, w module `flu_get_data` jest jeszcze funkcja `collect_all`, która dla lat `year`: 2025..2027 i dla wszystkich województw `region` wygeneruje pliki CSV:

* Z danymi z roku `year-1` dla modelu `forecast = flu_predict(year, region, infected, vaccinated, weather)`, prognozującego na rok `year`:
    * `Data/{year}-{region}-infected-data.csv` - czyli to, trzeba przekazać jako parametr `infected`
    * `Data/{year}-{region}-vaccinated-data.csv` - tzn. `vaccinated`
    * `Data/{year}-{region}-weather-data.csv` - tzn. `weather`
* Z danymi o faktycznej liczbie zachorowań w interesującym nas okresie roku `year`:
    * `Data/{year}-{region}-infected-forecast.csv`

Jak pamiętasz, dla `year` równego 2027, część danych nie będzie jeszcze dostępna.


In [11]:
fgd.collect_all()

Pobieram pełny arkusz Excela z portalu ezdrowie.gov.pl... (to może potrwać)
Pobieram współrzędne dla miasta Wrocław z portalu open-meteo.com
Pobieram współrzędne dla miasta Toruń z portalu open-meteo.com
Pobieram współrzędne dla miasta Lublin z portalu open-meteo.com
Pobieram współrzędne dla miasta Zielona Góra z portalu open-meteo.com
Pobieram współrzędne dla miasta Łódź z portalu open-meteo.com
Pobieram współrzędne dla miasta Kraków z portalu open-meteo.com
Pobieram współrzędne dla miasta Warszawa z portalu open-meteo.com
Pobieram współrzędne dla miasta Opole z portalu open-meteo.com
Pobieram współrzędne dla miasta Rzeszów z portalu open-meteo.com
Pobieram współrzędne dla miasta Białystok z portalu open-meteo.com
Pobieram współrzędne dla miasta Gdańsk z portalu open-meteo.com
Pobieram współrzędne dla miasta Katowice z portalu open-meteo.com
Pobieram współrzędne dla miasta Kielce z portalu open-meteo.com
Pobieram współrzędne dla miasta Olsztyn z portalu open-meteo.com
Pobieram współrz

Teraz już możesz przeprowadzić bardzo wiele testów swojego modelu.

*Udanego modelowania! I programowania też.*